# DATA209 — Advanced Exploratory Data Analysis
# Practical P15-16 · Data quality audit

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 8 · Module 3 · CO3

---

**Objective.** Audit the dataset against the six quality dimensions, remove duplicates, standardise inconsistent encodings, and produce a cleaning log.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 and P3-4 — the dataset, the split, and the corrected data types.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P3-4: corrected dtypes
dfc = df.copy()
for c in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    dfc[c] = dfc[c].astype("category")
_order  = ["Jan","Feb","Mar","Apr","May","June","Jul","Aug","Sep","Oct","Nov","Dec"]
_present = [m for m in _order if m in dfc["Month"].unique()]
dfc["Month"] = pd.Categorical(dfc["Month"], categories=_present, ordered=True)
dfc["VisitorType"] = dfc["VisitorType"].astype(str).str.strip()
for c in ["Weekend", "Revenue"]:
    dfc[c] = dfc[c].astype(bool)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P15-16 — Data quality audit

### Data quality audit

Six dimensions, audited in this order — validity failures often explain completeness failures,
which in turn explain accuracy failures.

| Dimension | Question |
|---|---|
| Accuracy | Do the values reflect reality? |
| Completeness | Is anything absent that should be present? |
| Consistency | Do values agree across records and sources? |
| Validity | Do values conform to type, range and set rules? |
| Uniqueness | Is each real-world entity represented once? |
| Timeliness | Is the data current, and from one period? |

In [ ]:
# ---- The profile table --------------------------------------------------
def quality_profile(frame):
    prof = pd.DataFrame({
        "dtype"     : frame.dtypes.astype(str),
        "non_null"  : frame.notna().sum(),
        "nulls"     : frame.isna().sum(),
        "null_%"    : (frame.isna().mean() * 100).round(2),
        "unique"    : frame.nunique(),
        "unique_%"  : (frame.nunique() / len(frame) * 100).round(2),
    })
    num = frame.select_dtypes(include=[np.number])
    prof["zeros_%"] = (num == 0).mean().mul(100).round(2)
    prof["min"]     = num.min()
    prof["max"]     = num.max()
    prof["constant"] = frame.nunique() <= 1
    return prof

profile = quality_profile(df)
print(profile.to_string())

print("\nCompleteness: total nulls =", int(df.isna().sum().sum()))
print("Note that zero nulls does NOT mean zero missing values — see P17-18.")

In [ ]:
# ---- Validity rules from domain knowledge -------------------------------
# Write the rules BEFORE looking at the data, then measure the violations.
rules = {
    "BounceRates"            : (0.0, 1.0),      # it is a rate
    "ExitRates"              : (0.0, 1.0),      # it is a rate
    "SpecialDay"             : (0.0, 1.0),      # closeness indicator
    "PageValues"             : (0.0, np.inf),   # cannot be negative
    "Administrative"         : (0, np.inf),     # a count
    "Informational"          : (0, np.inf),
    "ProductRelated"         : (1, np.inf),     # a session must view >= 1 page
    "Administrative_Duration": (0.0, np.inf),   # seconds
    "Informational_Duration" : (0.0, np.inf),
    "ProductRelated_Duration": (0.0, np.inf),
}

violations = []
for col, (lo, hi) in rules.items():
    bad = ~df[col].between(lo, hi)
    violations.append({"column": col, "rule": f"[{lo}, {hi}]",
                       "violations": int(bad.sum()),
                       "pct": round(bad.mean() * 100, 3)})
viol = pd.DataFrame(violations)
print(viol.to_string(index=False))

# cross-field rule: a duration cannot be positive when the page count is zero
cross = pd.DataFrame({
    "rule": ["Administrative_Duration > 0 while Administrative == 0",
             "Informational_Duration > 0 while Informational == 0",
             "ProductRelated_Duration > 0 while ProductRelated == 0"],
    "violations": [
        int(((df["Administrative"] == 0) & (df["Administrative_Duration"] > 0)).sum()),
        int(((df["Informational"] == 0) & (df["Informational_Duration"] > 0)).sum()),
        int(((df["ProductRelated"] == 0) & (df["ProductRelated_Duration"] > 0)).sum()),
    ]})
print("\nCross-field rules")
print(cross.to_string(index=False))

print("\nNegative durations, if any, are sentinel values or clock errors — never real.")

### Duplicate removal

Two kinds matter: **exact duplicates** across all columns, and **key duplicates** where the same
entity appears with differing fields. This dataset has no session id, so an exact duplicate is
either a genuine repeat of identical behaviour or a logging fault — a judgement call you must
record either way.

In [ ]:
# ---- Duplicates ---------------------------------------------------------
exact = df.duplicated()
print(f"Exact duplicate rows: {exact.sum():,} ({exact.mean()*100:.2f}%)")

dup_rows = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print("\nA sample of duplicated rows:")
print(dup_rows.head(6).to_string())

print("\nHow duplicates distribute across the target:")
print(df[df.duplicated(keep=False)][TARGET].value_counts(normalize=True).round(3).to_string())

df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"\nBefore: {len(df):,} rows   After: {len(df_clean):,} rows   "
      f"Removed: {len(df) - len(df_clean):,}")

print("\nEffect on the headline statistic:")
print(f"  conversion rate before {df[TARGET].mean()*100:.3f}%  "
      f"after {df_clean[TARGET].mean()*100:.3f}%")
print("A negligible change means duplicates were not driving the result — record that finding.")

### Standardization

Standardising here means making the *encodings* consistent — case, whitespace, category labels,
units. (Standardising the *scale* of numeric variables is a different operation, covered in P23-24.)

In [ ]:
# ---- Standardise categorical encodings ---------------------------------
def standardise_categories(frame, cols):
    """Trim whitespace, collapse internal spaces, and unify case for label columns."""
    out, log = frame.copy(), []
    for c in cols:
        before = out[c].nunique(dropna=False)
        s = out[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
        out[c] = s
        after = out[c].nunique(dropna=False)
        log.append({"column": c, "levels_before": before, "levels_after": after,
                    "collapsed": before - after})
    return out, pd.DataFrame(log)

df_clean, std_log = standardise_categories(df_clean, ["VisitorType", "Month"])
print(std_log.to_string(index=False))

print("\nValue set after standardisation:")
for c in ["VisitorType", "Month"]:
    print(f"  {c}: {sorted(df_clean[c].unique().tolist())}")

# demonstrate why it matters, on a deliberately dirtied copy
dirty = df[["VisitorType"]].head(200).copy()
dirty.loc[:60,  "VisitorType"] = " Returning_Visitor"
dirty.loc[61:120,"VisitorType"] = "returning_visitor "
print(f"\nDirtied sample: {dirty['VisitorType'].nunique()} apparent levels")
fixed = dirty["VisitorType"].str.strip().str.lower()
print(f"After trim + case-fold: {fixed.nunique()} real levels")
print("Every frequency table computed before this step would have been wrong.")

In [ ]:
# ---- Rare-level grouping: prepare high-cardinality columns -------------
def group_rare(series, threshold=0.01, other="Other"):
    share = series.value_counts(normalize=True)
    rare  = share[share < threshold].index
    return series.where(~series.isin(rare), other), list(rare)

for c in ["TrafficType", "Browser", "OperatingSystems", "Region"]:
    grouped, rare = group_rare(df_clean[c].astype(str))
    print(f"{c:18} {df_clean[c].nunique():3} levels -> {grouped.nunique():3} "
          f"({len(rare)} rare levels grouped into 'Other')")
    df_clean[c + "_grouped"] = grouped

print("\nThreshold used: levels below 1% of rows. Record the threshold — it is a parameter.")

In [ ]:
# ---- The cleaning log ---------------------------------------------------
cleaning_log = pd.DataFrame([
    dict(step=1, dimension="Uniqueness", finding="Exact duplicate rows",
         rows=int(exact.sum()), action="Dropped",
         justification="Identical across all 18 columns; no session id to distinguish them"),
    dict(step=2, dimension="Consistency", finding="Whitespace/case in label columns",
         rows=0, action="Trimmed and collapsed whitespace",
         justification="Prevents the same category being counted as two"),
    dict(step=3, dimension="Validity", finding="Rate columns checked against [0,1]",
         rows=int(viol.loc[viol.column.isin(['BounceRates','ExitRates']), 'violations'].sum()),
         action="Verified, none out of range",
         justification="Domain rule: a rate cannot exceed 1"),
    dict(step=4, dimension="Consistency", finding="Rare categorical levels",
         rows=0, action="Grouped below 1% into 'Other'",
         justification="Avoids column explosion and unstable estimates at encoding"),
    dict(step=5, dimension="Completeness", finding="No explicit nulls",
         rows=0, action="Flagged for P17-18",
         justification="Zero nulls does not prove zero missingness"),
])
print(cleaning_log.to_string(index=False))

print(f"\nFinal cleaned shape: {df_clean.shape}")
print("The cleaning log is a deliverable. It carries most of the marks in Assignment 2.")

### Assignment 1 review

Use this session to review Assignment 1 against the marking scheme:

| Criterion | Weight | What is being judged |
|---|---|---|
| Correctness of technique | 30% | Right method for the variable type; no Pearson on nominal data |
| Interpretation | 35% | Every figure carries a sentence saying what it means |
| Reproducibility | 15% | Runs top to bottom; relative paths; commented; seeded |
| Presentation | 20% | Clear narrative, honest about limits |

**The most common failures:** plots with no written interpretation; averaging ordinal or nominal
codes; absolute paths that only work on the author's machine; and reporting accuracy on an
imbalanced target.

### Deliverable — P15-16

A notebook plus the **cleaning log as a table** — finding, rows affected, action, justification —
with before/after counts for every step.